# FaceGuard v2 — FRAD ArcFace Embedding Extraction (GPU)
Audits the private FRAD legacy crops and runs pretrained ArcFace inference; it does **not** train ArcFace. No-face crops are retried on recorded neutral padded canvases, while multi-face crops remain rejected rather than guessing an identity.

In [ ]:
# --- 1. Clean Dependencies & GPU Setup ---
import os, sys, subprocess, zipfile
from pathlib import Path

print("Configuring clean OpenCV and installing GPU dependencies...")
# Completely wipe conflicting OpenCV variants and install clean headless OpenCV with InsightFace
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "opencv-python", "opencv-python-headless"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless", "insightface==2.1", "onnxruntime-gpu", "PyYAML"], check=True)

import cv2
print(f"[✓] OpenCV installed and verified: CV_8U={hasattr(cv2, 'CV_8U')}")

WORK_ROOT = Path("/kaggle/working/faceguard")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# --- 2. Locate Dataset Root ---
dataset_candidates = [
    Path("/kaggle/input/datasets/jit007/frad-context-crops-v2/frad_context_crops_v2"),
    Path("/kaggle/input/datasets/jit007/frad-context-crops-v2"),
    Path("/kaggle/input/frad-context-crops-v2/frad_context_crops_v2"),
    Path("/kaggle/input/frad-context-crops-v2"),
    Path("/kaggle/input/datasets/jit007/frad-legacy-crops/frad_legacy_crops"),
    Path("/kaggle/input/datasets/jit007/frad-legacy-crops"),
    Path("/kaggle/input/frad-legacy-crops/frad_legacy_crops"),
    Path("/kaggle/input/frad-legacy-crops"),
]
def resolve_face_root(path):
    for candidate in (path, path / "frad_context_crops_v2", path / "frad_legacy_crops"):
        if (candidate / "0001").is_dir():
            return candidate
    return None

DATASET_ROOT = None
for p in dataset_candidates:
    if p.exists():
        resolved = resolve_face_root(p)
        if resolved is not None:
            DATASET_ROOT = resolved
            break
        zips = list(p.glob("*.zip"))
        if zips:
            target_unzip = WORK_ROOT / "data" / "recognition_input"
            target_unzip.mkdir(parents=True, exist_ok=True)
            print(f"Unzipping {zips[0]}...")
            with zipfile.ZipFile(zips[0], "r") as zf:
                zf.extractall(target_unzip)
            DATASET_ROOT = resolve_face_root(target_unzip)
            if DATASET_ROOT is None:
                raise RuntimeError(f"ZIP does not contain a folder-per-person dataset: {zips[0]}")
            break

if DATASET_ROOT is None and Path("/kaggle/input").exists():
    for root, dirs, files in os.walk("/kaggle/input"):
        if "0001" in dirs:
            DATASET_ROOT = Path(root)
            break
        for f in files:
            if "frad" in f.lower() and f.endswith(".zip"):
                target_unzip = WORK_ROOT / "data" / "recognition_input"
                target_unzip.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(Path(root) / f, "r") as zf:
                    zf.extractall(target_unzip)
                DATASET_ROOT = resolve_face_root(target_unzip)
                if DATASET_ROOT is None:
                    raise RuntimeError(f"ZIP does not contain a folder-per-person dataset: {Path(root) / f}")
                break
        if DATASET_ROOT: break

if DATASET_ROOT is None:
    raise RuntimeError("No supported FRAD recognition dataset was found under /kaggle/input")
print(f"[✓] DATASET_ROOT: {DATASET_ROOT}")

# --- 3. Locate Code Bundle ---
code_candidates = [
    Path("/kaggle/input/datasets/jit007/faceguard-kaggle-source"),
    Path("/kaggle/input/faceguard-kaggle-source"),
]
PROJECT_ROOT = None
for p in code_candidates:
    if (p / "src" / "faceguard").is_dir():
        PROJECT_ROOT = p
        break
    zips = list(p.glob("*.zip")) if p.exists() else []
    if zips:
        target_code = WORK_ROOT / "code"
        target_code.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zips[0], "r") as zf:
            zf.extractall(target_code)
        PROJECT_ROOT = target_code
        break

if PROJECT_ROOT is None and Path("/kaggle/input").exists():
    for root, dirs, files in os.walk("/kaggle/input"):
        if "faceguard" in dirs and Path(root).name == "src":
            PROJECT_ROOT = Path(root).parent
            break
        for f in files:
            if "faceguard" in f.lower() and f.endswith(".zip"):
                target_code = WORK_ROOT / "code"
                target_code.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(Path(root) / f, "r") as zf:
                    zf.extractall(target_code)
                PROJECT_ROOT = target_code
                break
        if PROJECT_ROOT: break

print(f"[✓] PROJECT_ROOT: {PROJECT_ROOT}")

# --- 4. Download and Unpack buffalo_m Models ---
model_pack_dir = WORK_ROOT / "models" / "buffalo_m"
model_pack_dir.mkdir(parents=True, exist_ok=True)
if not (model_pack_dir / "w600k_r50.onnx").exists():
    print("Downloading official buffalo_m ONNX model pack...")
    zip_tmp = WORK_ROOT / "buffalo_m.zip"
    subprocess.run([
        "curl", "-L", "-s",
        "https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_m.zip",
        "-o", str(zip_tmp)
    ], check=True)
    with zipfile.ZipFile(zip_tmp, "r") as zf:
        zf.extractall(model_pack_dir)
    nested = model_pack_dir / "buffalo_m"
    if nested.exists() and nested.is_dir():
        for f in nested.iterdir():
            dest = model_pack_dir / f.name
            if not dest.exists(): f.rename(dest)
    if zip_tmp.exists(): zip_tmp.unlink()

print("[✓] Models in place:", [f.name for f in model_pack_dir.glob("*.onnx")])


In [ ]:
# --- 5. Structural Dataset Audit ---
INVENTORY_PATH = WORK_ROOT / "manifests/frad_inventory.csv"
SUBJECTS_PATH = WORK_ROOT / "manifests/frad_subjects.csv"
AUDIT_REPORT = WORK_ROOT / "reports/frad_audit.json"

print("--- Running Dataset Audit ---")
subprocess.run([
    sys.executable, str(PROJECT_ROOT / "scripts/audit_data.py"),
    "--dataset-root", str(DATASET_ROOT),
    "--inventory", str(INVENTORY_PATH),
    "--subjects", str(SUBJECTS_PATH),
    "--report", str(AUDIT_REPORT),
    "--overwrite"
], check=True)
print("[✓] Audit completed successfully!")


In [ ]:
# --- 6. ArcFace Embedding Extraction (Auto GPU/CPU) ---
import onnxruntime as ort

available = ort.get_available_providers()
print(f"Available ONNX Execution Providers: {available}")

if "CUDAExecutionProvider" in available:
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    print("[✓] Running ArcFace on GPU (CUDAExecutionProvider)")
else:
    providers = ["CPUExecutionProvider"]
    print("[✓] CUDAExecutionProvider not detected in ONNX Runtime; running on CPU (takes ~50s for 2,800 images)")

EMBEDDING_OUTPUT = WORK_ROOT / "private_embeddings"
DECISIONS_PATH = PROJECT_ROOT / "configs/decisions.yaml"

print("--- Extracting ArcFace Embeddings ---")
cmd = [
    sys.executable, str(PROJECT_ROOT / "scripts/extract_gallery.py"),
    "--dataset-root", str(DATASET_ROOT),
    "--manifest", str(INVENTORY_PATH),
    "--decisions", str(DECISIONS_PATH),
    "--model-root", str(WORK_ROOT),
    "--output-dir", str(EMBEDDING_OUTPUT),
    "--context", "kaggle",
    "--mode", "audit",
    "--pack", "buffalo_m",
    "--retry-padding-ratios", "0.25", "0.50",
    "--providers", *providers,
    "--acknowledge-non-commercial-model-terms",
    "--overwrite"
]
subprocess.run(cmd, check=True)
print("[✓] Embedding extraction finished!")


In [ ]:
# --- 7. Verify Output Artifacts ---
import json
import numpy as np
emb_file = EMBEDDING_OUTPUT / "frad_embeddings.npz"
metadata_file = EMBEDDING_OUTPUT / "frad_embedding_metadata.csv"
report_file = EMBEDDING_OUTPUT / "embedding_extraction.json"
if emb_file.exists():
    with np.load(emb_file, allow_pickle=False) as data:
        print(f"\n🎉 SUCCESS! Generated {emb_file.name} ({emb_file.stat().st_size / (1024*1024):.2f} MB)")
        for key in data.files:
            print(f"  - Array {key}: shape {data[key].shape}, dtype {data[key].dtype}")
    report = json.loads(report_file.read_text())
    print("\nExtraction outcomes:", report["outcomes"])
    print("Recovered by padding:", report["rescued_samples"])
    print("Embedded by input variant:", report["embedded_by_input_variant"])
    print(f"\nDownload all three restricted artifacts:\n  {emb_file}\n  {metadata_file}\n  {report_file}")
else:
    print("Error: Output file not found.")


## Important: extraction is not training or evaluation

The NPZ is a candidate embedding pool made by a fixed pretrained model. Extracting every image does not fit ArcFace and does not consume a test set. Before recognition evaluation, freeze separate manifests for enrollment, known validation probes, validation impostors, known test probes, and untouched test impostors. Choose thresholds using validation only; open the test manifests once after the gallery and threshold are frozen.

Do not create a random image split when several near-adjacent photos may come from the same capture session. Use verified session/device metadata where available. Without it, label results as same/unknown-session exploratory rather than cross-session performance.